# v.in.ags – Importing ArcGIS Server Data into GRASS GIS

This notebook demonstrates how to use the **v.in.ags** GRASS addon to download
vector features directly from an ArcGIS Server (AGS) REST API and import them
as GRASS vector maps.

**What you will learn:**

1. Start a GRASS session inside Jupyter
2. List layers available in an AGS feature service
3. Perform a basic import of all features
4. Filter features by attribute (SQL WHERE clause)
5. Filter features by bounding box (spatial extent)
6. Import only selected attribute fields
7. Reproject data to the project CRS on import
8. Inspect and visualise the imported map

**Requirements:**

- GRASS GIS 8.3 or later
- The `grass` Python package (`pip install grass`)
- The `v.in.ags` addon installed (`g.extension extension=v.in.ags`)
- An internet connection to reach the sample ArcGIS Server

## 1. Start a GRASS session

We create a temporary GRASS project in the WGS84 geographic coordinate system
(EPSG:4326). For production workflows you would use your own project.

In [3]:
import subprocess
import sys

# check where GRASS Python packages are and add them to path
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

In [4]:
import tempfile
import os
from pathlib import Path

# grass.jupyter provides a convenient session manager
import grass.jupyter as gj
import grass.script as gs

# Create a temporary GRASS project (WGS84) for this tutorial
tempdir = tempfile.TemporaryDirectory()

# Create a project in the temporary directory
gs.create_project(path=tempdir.name, name="wgs84_project", epsg="4326")
# Start GRASS in the recently created project
session = gj.init(Path(tempdir.name, "wgs84_project"))
print("GRASS session started")

GRASS session started


## 2. Install the addon (if not already installed)

In [12]:
# Install v.in.ags from the GRASS addons repository.
# Skip this cell if you have already installed the addon.
# gs.run_command("g.extension", extension="v.in.ags")
gs.run_command(
    "g.extension", extension="v.in.ags", url=str(Path().parent.cwd().as_uri())
)

## 3. List available layers in a service

Before importing, use the **-l** flag to explore which layers a service
exposes. We use Esri's public sample server here.

In [15]:
SERVICE_ROOT = (
    "https://sampleserver6.arcgisonline.com/arcgis/rest/services/USA/MapServer"
)

gs.run_command("v.in.ags", flags="l", url=SERVICE_ROOT)

The output lists each layer's numeric **ID**, its **type**, and its **name**.
Use the ID with the `layer` parameter when importing from a service root URL.

## 4. Basic import – all features from a layer

Import all features from layer 0 (cities) of the sample MapServer.
No reprojection is applied; the output map will be in WGS84.

In [16]:
CITIES_URL = SERVICE_ROOT + "/0"  # layer 0 = cities

gs.run_command(
    "v.in.ags",
    url=CITIES_URL,
    output="usa_cities",
    overwrite=True,
)

# Confirm import
info = gs.vector_info_topo("usa_cities")
print("Points imported:", info["points"])

Points imported: 3557


## 5. Attribute filter – SQL WHERE clause

Import only cities in California. The **where** parameter accepts any SQL
expression supported by the server.

In [17]:
gs.run_command(
    "v.in.ags",
    url=CITIES_URL,
    output="ca_cities",
    where="st = 'CA'",
    overwrite=True,
)

info = gs.vector_info_topo("ca_cities")
print("California cities imported:", info["points"])

California cities imported: 446


## 6. Spatial filter – bounding box

The **extent** option restricts the download to features that intersect a
bounding box. Coordinates must be in WGS84 (decimal degrees):
`xmin,ymin,xmax,ymax`.

In [18]:
# Bounding box covering the US Pacific Northwest
PNW_EXTENT = "-125,42,-116,49"

gs.run_command(
    "v.in.ags",
    url=CITIES_URL,
    output="pnw_cities",
    extent=PNW_EXTENT,
    overwrite=True,
)

info = gs.vector_info_topo("pnw_cities")
print("Pacific Northwest cities imported:", info["points"])

Pacific Northwest cities imported: 158


## 7. Combine attribute and spatial filters

In [19]:
# Large cities (population > 100 000) anywhere in the continental US
CONUS_EXTENT = "-125,24,-66,50"

gs.run_command(
    "v.in.ags",
    url=CITIES_URL,
    output="large_cities",
    where="pop2000 > 100000",
    extent=CONUS_EXTENT,
    overwrite=True,
)

info = gs.vector_info_topo("large_cities")
print("Large cities imported:", info["points"])

Large cities imported: 243


## 8. Selective field import

Use **fields** to download only a subset of attributes. This reduces
transfer size for wide tables.

In [20]:
gs.run_command(
    "v.in.ags",
    url=CITIES_URL,
    output="cities_slim",
    fields="areaname,st,pop2000",
    overwrite=True,
)

# Show column names
columns = gs.read_command("v.info", map="cities_slim", flags="c")
print(columns)

INTEGER|cat
TEXT|areaname
TEXT|st
INTEGER|pop2000



## 9. Reproject on import with -r flag

If your GRASS project uses a projected CRS (e.g. UTM), pass **-r** so that
*v.import* reprojects the WGS84 GeoJSON into the project CRS automatically.

Here we demonstrate by creating a UTM project and importing into it.

In [ ]:
# Create a second project in UTM Zone 10N (EPSG:32610)
utm_project = os.path.join(tmp_dir, "utm10n_project")
utm_session = gj.init(utm_project, epsg=32610)

gs.run_command(
    "v.in.ags",
    flags="r",  # reproject using v.import
    url=CITIES_URL,
    output="pnw_cities_utm",
    extent=PNW_EXTENT,
    overwrite=True,
)

# Verify: projection info should show UTM Zone 10N
proj = gs.read_command("v.proj", input="pnw_cities_utm", flags="g")
print(gs.parse_command("g.proj", flags="g"))

## 10. Visualise the imported data

Switch back to the WGS84 project and use `grass.jupyter.Map` to display
the imported cities layer.

In [ ]:
# Return to the WGS84 session
session = gj.init(project_path, epsg=4326)

# Set the region to match the large_cities extent
gs.run_command("g.region", vector="large_cities", grow=2)

# Display with grass.jupyter
m = gj.Map()
m.d_background(color="white")
m.d_vect(
    map="large_cities", icon="basic/circle", size=8, color="blue", fill_color="cyan"
)
m.d_vect_thematic(
    map="large_cities",
    column="pop2000",
    legend_label="Population (2000)",
)
m.d_grid(size=5, color="grey")
m.d_legend_vect(flags="b")
m.show()

## 11. Post-import analysis

Once data is in GRASS, the full suite of GRASS vector tools is available.

In [ ]:
# Basic attribute statistics on imported population field
stats = gs.parse_command(
    "v.univar",
    map="large_cities",
    column="pop2000",
    type="point",
    flags="g",
)

print("Count :", stats["n"])
print("Min   :", stats["min"])
print("Max   :", stats["max"])
print("Mean  :", stats["mean"])

In [ ]:
# Query the top 5 cities by population
top5 = gs.read_command(
    "v.db.select",
    map="large_cities",
    columns="areaname,st,pop2000",
    where="pop2000 > 500000",
    format="csv",
)
print(top5)

## 12. Using v.in.ags in a reproducible workflow

The snippet below shows how to embed **v.in.ags** in a larger analysis
pipeline – importing data, rasterising it, and computing zonal statistics.

In [ ]:
# Example pipeline (requires a raster elevation map in the project)
# -------------------------------------------------------------------
# 1. Import county boundaries for California

COUNTIES_URL = (
    "https://services.arcgis.com/P3ePLMYs2RVChkJx"
    "/arcgis/rest/services/USA_Counties_Generalized/FeatureServer/0"
)

gs.run_command(
    "v.in.ags",
    url=COUNTIES_URL,
    output="ca_counties",
    where="STATE_NAME = 'California'",
    fields="NAME,STATE_NAME,POP2020",
    overwrite=True,
)

# 2. Show the result
info = gs.vector_info_topo("ca_counties")
print("California counties imported:", info["areas"])

## 13. Clean up

In [ ]:
import shutil

# Remove temporary GRASS projects
shutil.rmtree(tmp_dir, ignore_errors=True)
print("Temporary projects removed.")

## Summary

| Task | Command |
|------|---------|
| List layers | `v.in.ags -l url=<service_root>` |
| Import all features | `v.in.ags url=<layer_url> output=<name>` |
| Attribute filter | `... where="field = 'value'"` |
| Spatial filter | `... extent="xmin,ymin,xmax,ymax"` |
| Selective fields | `... fields="col1,col2"` |
| Reproject to project CRS | `v.in.ags -r url=... output=...` |

For full documentation see `v.in.ags --help` or the
[GRASS addons manual](https://grass.osgeo.org/grass-devel/manuals/addons/v.in.ags.html).